# Data-minimizing tool-calling agents: full experiment

Runs the whole Track 3 experiment on Tiny Aya Global: baselines, two AgentTune fine-tunes, AuditKIT evaluation, comparison and summary.

**Before you start:** Runtime > Change runtime type > a GPU. The reported results were produced on an **NVIDIA L4** (Colab); a T4 is too small for a 3.35B model unless weights are loaded in fp16 (the training script does that automatically, but it is slower).
Fine-tuned models are evaluated after merging the LoRA adapter and fixing the stop tokens (step 6b), which makes evaluation much faster.

## 1. Get the kit
Upload `data-minimizing-agent.zip` when asked (or drop it in the file browser first). If you cloned the repo from GitHub instead, run `!git clone <your repo url>` and `%cd` into it, then continue with step 2.

In [ ]:
import os, zipfile
os.chdir("/content")
if not os.path.exists("data-minimizing-agent"):
    if not os.path.exists("data-minimizing-agent.zip"):
        from google.colab import files
        files.upload()
    zipfile.ZipFile("data-minimizing-agent.zip").extractall(".")
os.chdir("/content/data-minimizing-agent")
print(os.getcwd(), os.listdir("."))

## 2. Install AgentTune and AuditKIT
AgentTune needs Python 3.12+ and pins its own PyTorch, so this replaces Colab's PyTorch and takes several minutes.

**When it finishes: Runtime > Restart session, then continue from step 3.** Do not run this cell again after restarting.

In [ ]:
import sys
print(sys.version)
!pip install -q agenttune "auditkit[transformers]"

## 3. After the restart: check the setup
All three commands must end with a success line before you spend GPU time.

In [ ]:
import os
os.chdir("/content/data-minimizing-agent")
os.makedirs("results", exist_ok=True)
import torch, agenttune, auditkit
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| agenttune", agenttune.__version__, "| auditkit", auditkit.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python tests/sanity_check.py | tail -n 1
!python tests/smoke_test_pipeline.py 2>&1 | tail -n 1

## 4. Settings
- `MODEL`: the model to fine-tune. The hackathon requires a Cohere Labs model for the core submission. `CohereLabs/tiny-aya-global` (3.35B) is the default. It is **gated**: accept its licence on its Hugging Face page and add your token (see the cell below) before running anything.
- `LIMIT`: number of test samples per split. `None` = all (200 in-distribution + 100 held-out tools). Use `60` for a quick first pass.
- `TWINS`: run the counterfactual test too. It doubles evaluation time.

In [ ]:
MODEL = "CohereLabs/tiny-aya-global"
LIMIT = None   # set to 60 for a quick first pass
TWINS = True

def evaluate(model, label, system="plain", baseline=None, batch_size=None):
    for split in ("test_id", "test_ood"):
        cmd = (f"python scripts/eval_auditkit.py --model {model} --system {system} --data data/{split}.jsonl "
               f"--label {label} --out results/{label}_{split}.json")
        if TWINS:
            cmd += f" --twins data/{split}.twins.jsonl"
        if LIMIT:
            cmd += f" --limit {LIMIT}"
        if baseline:
            cmd += f" --compare-to results/{baseline}_{split}.auditkit.json"
        if batch_size:
            cmd += f" --batch-size {batch_size}"
        print("$", cmd)
        !{cmd}


In [ ]:
# Hugging Face login (needed because Tiny Aya is gated).
# 1) Open https://huggingface.co/CohereLabs/tiny-aya-global while logged in and accept the licence.
# 2) Create a read token at https://huggingface.co/settings/tokens
# 3) In Colab: left sidebar > key icon > add a secret named HF_TOKEN with that token, turn on Notebook access.
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login, model_info
login(os.environ["HF_TOKEN"])
print(model_info(MODEL).id, "is reachable")


## 5. Baselines: the untouched model, without and with a privacy instruction
Arm A (`base`) and arm B (`base_privacy`). B is the baseline your fine-tune has to beat.

In [ ]:
evaluate(f"hf:{MODEL}", "base")
evaluate(f"hf:{MODEL}", "base_privacy", system="privacy")

## 6. Fine-tune with AgentTune: two arms that differ only in the demonstrations
- `runs/overfilled` (arm C): the same correct calls, but with unneeded personal data in optional arguments. This stands in for ordinary agentic fine-tuning on "helpful" data. **Do not publish this adapter.**
- `runs/minimal` (arm D): calls carry only what the tool needs.

Each arm took about 27 to 29 minutes on an L4 with the defaults (3 epochs, 600 demonstrations, LoRA rank 16). If you hit an out-of-memory error try `--max-length 1024`. If the loss is `nan` add `--dtype float32` (only if it fits).

In [ ]:
!python scripts/train_agenttune.py --model {MODEL} --leak-frac 1.0 --out runs/overfilled
!python scripts/train_agenttune.py --model {MODEL} --leak-frac 0.0 --out runs/minimal

## 6b. Merge the adapters and fix the stop tokens
The fine-tuned model ends its answer with `<|END_OF_TURN_TOKEN|>`, but generation only stops on `<EOS_TOKEN>`, so it kept generating filler up to the token limit (about 4x slower evaluation, same content). `merge_fix.py` merges each adapter into the base model and tells generation to stop on both tokens.

In [ ]:
!python merge_fix.py runs/overfilled runs/overfilled_merged
!python merge_fix.py runs/minimal runs/minimal_merged

## 7. Evaluate both fine-tunes with AuditKIT
No privacy instruction is given to either. The merged models are evaluated in batches of 8 (the baselines above ran one at a time; greedy decoding means the content does not depend on this, but report it).

In [ ]:
evaluate("hf:runs/minimal_merged", "ft_minimal", baseline="base", batch_size=8)
evaluate("hf:runs/overfilled_merged", "ft_overfilled", baseline="base", batch_size=8)

## 8. Compare the four arms
One table and one chart per test split. `test_ood` uses two tools the fine-tunes never saw.

In [ ]:
from IPython.display import Image, Markdown, display
for split in ("test_id", "test_ood"):
    files_ = " ".join(f"results/{arm}_{split}.json" for arm in ("base", "base_privacy", "ft_overfilled", "ft_minimal"))
    !python scripts/compare.py {files_} --out results/comparison_{split} > /dev/null
    display(Markdown(open(f"results/comparison_{split}.md").read()))
    display(Image(f"results/comparison_{split}.png"))

## 9. Optional, NOT run for the reported results: one extra arm at 50% over-filled
A third point between 0% and 100%. About 45 to 60 minutes. Untested as a single cell; use the same training settings as the main arms.

In [ ]:
from IPython.display import Markdown, display
!python scripts/train_agenttune.py --model {MODEL} --leak-frac 0.5 --out runs/ft_mix50
!python merge_fix.py runs/ft_mix50 runs/ft_mix50_merged
evaluate("hf:runs/ft_mix50_merged", "ft_mix50", baseline="base", batch_size=8)
for split in ("test_id", "test_ood"):
    files_ = " ".join(f"results/{arm}_{split}.json" for arm in ("ft_minimal", "ft_mix50", "ft_overfilled"))
    !python scripts/compare.py {files_} --out results/dose_{split} > /dev/null
    display(Markdown(open(f"results/dose_{split}.md").read()))

## 10. Look at real outputs, summarise, then download everything
Read some raw calls before you trust the numbers. `final_summary.py` prints a verdict per hypothesis with bootstrap intervals and saves `results/WRITEUP_FACTS.md`. Never publish `runs/overfilled`.

In [ ]:
import json
r = json.load(open("results/ft_minimal_test_id.json"))
b = {x["id"]: x for x in json.load(open("results/base_test_id.json"))["records"]}
for rec in r["records"][:8]:
    print(rec["id"], rec["variant"], "| base leaked:", b[rec["id"]]["leaked_fields"], "| tuned leaked:", rec["leaked_fields"])
    print("  base :", (b[rec["id"]]["raw"] or "").strip()[:220])
    print("  tuned:", (rec["raw"] or "").strip()[:220])

In [ ]:
%run final_summary.py

In [ ]:
!zip -qr /content/results.zip results runs/*/dmin_run_info.json
!zip -qr /content/minimal_adapter.zip runs/minimal
from google.colab import files
files.download("/content/results.zip"); files.download("/content/minimal_adapter.zip")